# Streamlined Analysis Template

This notebook demonstrates how to use the new `ExperimentResults` and `AttributionAnalyzer` for EEG attribution analysis.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Add src to path
sys.path.append(os.path.abspath("../src"))

from evaluate.analysis_utils import ExperimentResults, AttributionAnalyzer
from visualization.eeg_attribution import attribution_topomap, plot_attribution_summary

import logging
import sys
import seaborn as sns

# 1. Setup the logger
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# 2. Configure basicConfig to use stdout
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    stream=sys.stdout
)

logger.info("This log should now appear in the notebook cell!")

## 1. Load Experiment Results
Point to your experiment directory (the one containing `run_*` folders and `evaluation_summary.csv`).

In [ ]:
EXPERIMENT_DIR = "$HOME/Downloads/true_binary_physionet_pretrained_evaluation"
results = ExperimentResults(EXPERIMENT_DIR)
analyzer = AttributionAnalyzer(results)

print(f"Discovered {len(results.run_dirs)} runs.")
if results.performance_df is not None:
    display(results.performance_df.head())

## 2. Calculate Global Statistics (Full Dataset)
Unlike previous notebooks, this uses **Dask** to process the entire dataset without sampling.

In [ ]:
RUN_IDX = 3
ddf = analyzer.load_run_data(RUN_IDX, split="test")

print("Calculating class-wise statistics...")
class_stats = analyzer.calculate_classwise_statistics(ddf)

for label, stats in class_stats.items():
    print(f"Class {label} ({stats.get('class_name')}): {stats['all_count']} examples")

In [ ]:
class_stats

## 3. Visualization
Use the aggregated statistics for plotting.

In [ ]:
# For further analysis, you may want to load the dataset corresponding to your run
from data.physionet_datamodule import PhysionetDatamodule
from utils import adapt_channel_names_to_mne
datamodule = PhysionetDatamodule(dataset_path="data/physiomi", data_dir="$HOME/Projects/datasets/physiomi", classes=['left_hand', 'right_hand'])
# datamodule.prepare_data()
datamodule.setup()
dataset = datamodule.test_ds
print(dataset)
ch_names = adapt_channel_names_to_mne(dataset[0]['eeg_channels'])
print(ch_names)

In [ ]:
# Example: Plot mean attribution for class 0
if 0 in class_stats:
    mean_attr = class_stats[0]["all_mean"]
    # Assuming you have channel names
    attribution_topomap(mean_attr, channel_names=ch_names, title="Mean Attribution for Class 0")
    plot_attribution_summary(mean_attr, ch_names)

In [ ]:
res_df = analyzer.load_run_data_to_memory(RUN_IDX, split="test")
res_df['subject_id'] = dataset['subject_id']
res_df['event_name'] =dataset['event_name']
res_df['run'] = [i.split('R')[1][:2] for i in dataset['run_file']]
res_df

In [ ]:
res_df['correct'] = res_df['prediction'] == res_df['label']
grouped = res_df.groupby(['subject_id', 'correct', 'run']).attribution.agg(average='mean', count='count')

In [ ]:
plot_attribution_summary(grouped.loc['S021', True, '04'].average, ch_names)
''

## 4. SpRAy Clustering
Identify decision strategies across the dataset.

In [ ]:
spray_res = analyzer.spray_clustering(ddf, n_clusters=5, downsample_factor=30)
print(f"Generated {spray_res['n_clusters']} clusters.")

In [ ]:
clustered_df =res_df
clustered_df['cluster'] = spray_res['labels']
for i in range(5):
    plot_attribution_summary(clustered_df.groupby('cluster').attribution.mean().loc[i], ch_names, title=f'Cluster {i}')


In [ ]:
clustered_df.groupby('cluster').correct.mean()

In [ ]:
clustered_df.groupby('cluster').run.value_counts(normalize=True)

In [ ]:
datamodule.class_to_idx['left_hand']

In [ ]:
clustered_df

In [ ]:
sns.displot(data=clustered_df, x='cluster', hue='subject_id', kind='hist')

In [ ]:
clustered_df

In [ ]:
sns.displot(data = clustered_df, x='correct', hue='cluster', kind='kde')